
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Using Delta Lake Features with Databricks SQL
In this demo, we’ll explore the powerful features of Delta Lake and demonstrate how they enhance data management in a data warehousing context. We’ll start by creating and exploring Delta tables, then dive into key features like Time Travel and Version History.

**Learning Objectives**

By the end of this demo, you will be able to:
- Explore key features of Delta Lake such as **Time Travel**, **Version History**, and metadata management.
- Use SQL commands like `DESCRIBE EXTENDED`, `DESCRIBE HISTORY`, `VERSION AS OF`, and `RESTORE TABLE`.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
    - In the drop-down, select **More**.
    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.
    
**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

- To run this notebook, you need to use one of the following Databricks runtime(s): `17.3.x-scala2.13`

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo.

In [0]:
%run ../Includes/Classroom-Setup-2

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
%python
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets.retail}")

Username:          labuser12730509_1763721946@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12730509_1763721946
Working Directory: /Volumes/dbacademy/ops/labuser12730509_1763721946@vocareum_com
Dataset Location:  /Volumes/dbacademy_retail/v01


## Introduction
Delta Lake is an open-source storage layer that brings reliability, performance, and ACID transactions to data lakes. In this demo, we will:
- Create and explore a Delta table.
- Simulate data changes and view metadata and history.
- Use Time Travel to query and restore data to previous states.

%md
## Setup

**Step 1: Select a default catalog and schema**
- Run the following commands to select a default catalog and schema, which makes referencing table names easier.

In [0]:
USE CATALOG ${DA.catalog_name};
USE SCHEMA ${DA.schema_name};

**Step 2: Create the `retail_sales` table** and view the data in the table
- We will use a pre-existing dataset file located at `DA.paths.datasets.retail/source_files/sales.csv`.

In [0]:
-- Create a Delta table from the CSV file
DROP TABLE IF EXISTS retail_sales;
CREATE TABLE IF NOT EXISTS retail_sales
USING DELTA
AS
SELECT *
FROM read_files(
  '${DA.paths.datasets.retail}/source_files/sales.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Display Data of the retail_sales table
SELECT * FROM retail_sales LIMIT 10;

customer_id,customer_name,product_name,order_date,product_category,product,total_price,_rescued_data
17372531,"RAMSEY, SHELBERT",Ramsung EVO+ 256GB UHS-I microSDXC U3 Memory Card with Adapter (MB-MC256DA/AM),2019-10-15,Ramsung,"""{""""curr"""":""""USD""""","""""id"""":""""AVpiE9hhilAPnD_xAfSU""""",null
58578517,"MILLER, LAWANDA Y",SP-FS52 Andrew Jones Designed Floorstanding Loudspeaker,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfHcah1cnluZ0-eQLY""""",null
58578517,"MILLER, LAWANDA Y",Sioneer GM-D8601 Class D Mono Amplifier with Wired Bass Boost Remote,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgRiy2LJeJML43Lk7h""""",null
17372531,"RAMSEY, SHELBERT","Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English""""""""",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpf-2hGilAPnD_xlfDv""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT",Cyber-shot DSC-RX100 V Digital Camera,2019-10-15,Rony,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfWGrYLJeJML437hk2""""",null
58578517,"MILLER, LAWANDA Y",Elite A-20 2-Channel Integrated Amplifier,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgUl_U1cnluZ0-z3Gz""""",null
58578517,"MILLER, LAWANDA Y",Opple MD825AM/A Lightning to VGA Adapter for iPhones,2019-08-07,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpggL_W1cnluZ0-2Wfp""""",null
58578517,"MILLER, LAWANDA Y",Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVz5wclz-jtxr-f30F66""""",null


## Part 1: Explore the Table

Delta Lake tables store metadata that can be queried for insights about the table structure, state, and history.

### Step 1: View Extended Metadata
Use the following command to view detailed metadata about the `retail_sales` table:

In [0]:
DESCRIBE EXTENDED retail_sales;

col_name,data_type,comment
customer_id,int,null
customer_name,string,null
product_name,string,null
order_date,date,null
product_category,string,null
product,string,null
total_price,string,null
_rescued_data,string,null
,,
# Delta Statistics Columns,,


### Step 2: Describe the History of the Table
Run the following command to view the history of `retail_sales`:

In [0]:
DESCRIBE HISTORY retail_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2025-11-21T11:44:39Z,77544492563404,labuser12730509_1763721946@vocareum.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3192803527457153),1121-104633-qjyc3doy,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 360, numOutputBytes -> 7570)",null,Databricks-Runtime/17.3.x-scala2.13


As the `retail_sales` table has not been modified yet, its version history will only show **Version 0**, which represents the initial creation of the table. Further modifications to the table will create additional versions in the history log.

The history includes detailed metadata about the table's state and modifications:
- **Version**: Indicates the specific version of the table.
- **Timestamp**: Specifies when the operation occurred.
- **Operation**: Describes the type of modification (e.g., `INSERT`, `UPDATE`, `DELETE`).
- **Operation Metrics**: Includes key metrics like the number of rows added, removed, or affected, and the total data size.
- **User Information**: Tracks which user performed the operation.
- **Cluster Information**: Logs the cluster ID where the operation was executed.

## Part 2: Simulating Data Changes and Time Travel Queries

Delta Lake allows for simulating data modifications and querying historical versions of the data using Time Travel.

###Simulate Data Changes
We will simulate updates and deletions to demonstrate how Delta Lake tracks changes in the table's version history.

#### Task 1: Update the Table
Use the following command to update the `retail_sales` table, updating the `product_name` column for all rows where the `product_category` is `Rony`.

In [0]:
UPDATE retail_sales
   SET product_name = 'Updated Items'
   WHERE product_category = 'Rony';

num_affected_rows
118


Before proceeding, let's obtain a timestamp representing this particular moment in time, which will be useful in a subsequent task. After executing, copy the resulting value to the clipboard for later use.

In [0]:
SELECT current_timestamp();

current_timestamp()
2025-11-21T11:44:47.016401Z


#### Task 2: Delete Records
Use the following command to delete specific records from the `retail_sales` table, removing all records for a specific customer:

In [0]:
DELETE FROM retail_sales
WHERE customer_name = 'VASQUEZ,  YVONNE M';

num_affected_rows
142


###View the Table's History

To understand the number of versions currently in the table, run the following command:

In [0]:
DESCRIBE HISTORY retail_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-11-21T11:44:51Z,77544492563404,labuser12730509_1763721946@vocareum.com,DELETE,"Map(predicate -> [""(customer_name#7070 = VASQUEZ, YVONNE M)""])",null,List(3192803527457153),1121-104633-qjyc3doy,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 2140, numDeletionVectorsUpdated -> 1, numDeletedRows -> 142, scanTimeMs -> 929, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 1211)",null,Databricks-Runtime/17.3.x-scala2.13
1,2025-11-21T11:44:46Z,77544492563404,labuser12730509_1763721946@vocareum.com,UPDATE,"Map(predicate -> [""(product_category#6537 = Rony)""])",null,List(3192803527457153),1121-104633-qjyc3doy,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2471, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1225, numAddedFiles -> 1, numUpdatedRows -> 118, numAddedBytes -> 4535, rewriteTimeMs -> 1223)",null,Databricks-Runtime/17.3.x-scala2.13
0,2025-11-21T11:44:39Z,77544492563404,labuser12730509_1763721946@vocareum.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3192803527457153),1121-104633-qjyc3doy,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 360, numOutputBytes -> 7570)",null,Databricks-Runtime/17.3.x-scala2.13


Delta Lake supports various operations that allow robust data management:
- **OPTIMIZE**: Optimizes the storage layout of the Delta table for better query performance. This operation compacts smaller files into larger ones, reducing the number of files scanned during queries.
- **UPDATE**: Modifies existing records in the table based on a condition.
- **DELETE**: Removes specific rows from the table based on a condition.

These operations enable efficient data management and ensure the table remains up-to-date with minimal manual intervention.

###Time Travel Queries

Delta Lake's Time Travel feature allows you to query data as it existed in previous versions or at specific timestamps.

#### Task 1: Query a Specific Version
Retrieve data from an early version of the table using the following command:

In [0]:
SELECT * 
  FROM retail_sales
  VERSION AS OF 0
  LIMIT 10;

customer_id,customer_name,product_name,order_date,product_category,product,total_price,_rescued_data
17372531,"RAMSEY, SHELBERT",Ramsung EVO+ 256GB UHS-I microSDXC U3 Memory Card with Adapter (MB-MC256DA/AM),2019-10-15,Ramsung,"""{""""curr"""":""""USD""""","""""id"""":""""AVpiE9hhilAPnD_xAfSU""""",null
58578517,"MILLER, LAWANDA Y",SP-FS52 Andrew Jones Designed Floorstanding Loudspeaker,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfHcah1cnluZ0-eQLY""""",null
58578517,"MILLER, LAWANDA Y",Sioneer GM-D8601 Class D Mono Amplifier with Wired Bass Boost Remote,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgRiy2LJeJML43Lk7h""""",null
17372531,"RAMSEY, SHELBERT","Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English""""""""",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpf-2hGilAPnD_xlfDv""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT",Cyber-shot DSC-RX100 V Digital Camera,2019-10-15,Rony,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfWGrYLJeJML437hk2""""",null
58578517,"MILLER, LAWANDA Y",Elite A-20 2-Channel Integrated Amplifier,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgUl_U1cnluZ0-z3Gz""""",null
58578517,"MILLER, LAWANDA Y",Opple MD825AM/A Lightning to VGA Adapter for iPhones,2019-08-07,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpggL_W1cnluZ0-2Wfp""""",null
58578517,"MILLER, LAWANDA Y",Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVz5wclz-jtxr-f30F66""""",null


#### Task 2: Query by Timestamp
Retrieve data as it existed at a specific timestamp. Replace the text below with the timestamp copied earlier, uncomment the following lines, and run the cell.

In [0]:
-- SELECT * 
--   FROM retail_sales
--   TIMESTAMP AS OF 'PASTE TIMESTAMP HERE';

## Part 3: Restore Table to a Previous Version

Delta Lake provides a powerful feature to restore a table to a previous state. This is particularly useful in scenarios where data is accidentally modified or deleted.

###View Table History
Before restoring, check the table's history to identify the version you want to restore:

In [0]:
DESCRIBE HISTORY retail_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2025-11-21T11:44:53Z,77544492563404,labuser12730509_1763721946@vocareum.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3192803527457153),1121-104633-qjyc3doy,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 12105, p25FileSize -> 7065, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 414, minFileSize -> 7065, numAddedFiles -> 1, maxFileSize -> 7065, p75FileSize -> 7065, p50FileSize -> 7065, numAddedBytes -> 7065)",null,Databricks-Runtime/17.3.x-scala2.13
2,2025-11-21T11:44:51Z,77544492563404,labuser12730509_1763721946@vocareum.com,DELETE,"Map(predicate -> [""(customer_name#7070 = VASQUEZ, YVONNE M)""])",null,List(3192803527457153),1121-104633-qjyc3doy,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 2140, numDeletionVectorsUpdated -> 1, numDeletedRows -> 142, scanTimeMs -> 929, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 1211)",null,Databricks-Runtime/17.3.x-scala2.13
1,2025-11-21T11:44:46Z,77544492563404,labuser12730509_1763721946@vocareum.com,UPDATE,"Map(predicate -> [""(product_category#6537 = Rony)""])",null,List(3192803527457153),1121-104633-qjyc3doy,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2471, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1225, numAddedFiles -> 1, numUpdatedRows -> 118, numAddedBytes -> 4535, rewriteTimeMs -> 1223)",null,Databricks-Runtime/17.3.x-scala2.13
0,2025-11-21T11:44:39Z,77544492563404,labuser12730509_1763721946@vocareum.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3192803527457153),1121-104633-qjyc3doy,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 360, numOutputBytes -> 7570)",null,Databricks-Runtime/17.3.x-scala2.13


This command displays the table's operation history, including timestamps and version numbers.


###Restore the Table
Use the following command to restore the `retail_sales` table to a specific version. For this demo, we’ll restore it to **Version 2**:

In [0]:
RESTORE TABLE retail_sales TO VERSION AS OF 2;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
12105,2,1,2,7065,12105


This command reverts the table to the specified version, undoing any changes made after that version.

###Verify the Restoration
After restoring, query the table to confirm that it has been reverted to the expected state:

In [0]:
SELECT * FROM retail_sales LIMIT 10;

customer_id,customer_name,product_name,order_date,product_category,product,total_price,_rescued_data
17372531,"RAMSEY, SHELBERT",Ramsung EVO+ 256GB UHS-I microSDXC U3 Memory Card with Adapter (MB-MC256DA/AM),2019-10-15,Ramsung,"""{""""curr"""":""""USD""""","""""id"""":""""AVpiE9hhilAPnD_xAfSU""""",null
58578517,"MILLER, LAWANDA Y",SP-FS52 Andrew Jones Designed Floorstanding Loudspeaker,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfHcah1cnluZ0-eQLY""""",null
58578517,"MILLER, LAWANDA Y",Sioneer GM-D8601 Class D Mono Amplifier with Wired Bass Boost Remote,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgRiy2LJeJML43Lk7h""""",null
17372531,"RAMSEY, SHELBERT","Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English""""""""",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpf-2hGilAPnD_xlfDv""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
58578517,"MILLER, LAWANDA Y",Elite A-20 2-Channel Integrated Amplifier,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgUl_U1cnluZ0-z3Gz""""",null
58578517,"MILLER, LAWANDA Y",Opple MD825AM/A Lightning to VGA Adapter for iPhones,2019-08-07,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpggL_W1cnluZ0-2Wfp""""",null
58578517,"MILLER, LAWANDA Y",Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVz5wclz-jtxr-f30F66""""",null
58578517,"MILLER, LAWANDA Y",Ramsung J3 - Verizon Prepaid,2019-08-07,Ramsung,"""{""""curr"""":""""USD""""","""""id"""":""""AVqVGHKQnnc1JgDc3jDC""""",null


The table will reflect the data as it existed in **Version 2**. This can also be validated by reviewing the history.
Use this feature to safeguard data integrity and recover from unintended modifications.


In [0]:
DESCRIBE HISTORY retail_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2025-11-21T11:45:49Z,77544492563404,labuser12730509_1763721946@vocareum.com,RESTORE,"Map(version -> 2, timestamp -> null)",null,List(3192803527457153),1121-104633-qjyc3doy,3,Serializable,false,"Map(numRestoredFiles -> 2, removedFilesSize -> 7065, numRemovedFiles -> 1, restoredFilesSize -> 12105, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 1, numOfFilesAfterRestore -> 2, tableSizeAfterRestore -> 12105)",null,Databricks-Runtime/17.3.x-scala2.13
3,2025-11-21T11:44:53Z,77544492563404,labuser12730509_1763721946@vocareum.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3192803527457153),1121-104633-qjyc3doy,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 12105, p25FileSize -> 7065, numDeletionVectorsRemoved -> 1, conflictDetectionTimeMs -> 414, minFileSize -> 7065, numAddedFiles -> 1, maxFileSize -> 7065, p75FileSize -> 7065, p50FileSize -> 7065, numAddedBytes -> 7065)",null,Databricks-Runtime/17.3.x-scala2.13
2,2025-11-21T11:44:51Z,77544492563404,labuser12730509_1763721946@vocareum.com,DELETE,"Map(predicate -> [""(customer_name#7070 = VASQUEZ, YVONNE M)""])",null,List(3192803527457153),1121-104633-qjyc3doy,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 2140, numDeletionVectorsUpdated -> 1, numDeletedRows -> 142, scanTimeMs -> 929, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 1211)",null,Databricks-Runtime/17.3.x-scala2.13
1,2025-11-21T11:44:46Z,77544492563404,labuser12730509_1763721946@vocareum.com,UPDATE,"Map(predicate -> [""(product_category#6537 = Rony)""])",null,List(3192803527457153),1121-104633-qjyc3doy,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2471, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1225, numAddedFiles -> 1, numUpdatedRows -> 118, numAddedBytes -> 4535, rewriteTimeMs -> 1223)",null,Databricks-Runtime/17.3.x-scala2.13
0,2025-11-21T11:44:39Z,77544492563404,labuser12730509_1763721946@vocareum.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(3192803527457153),1121-104633-qjyc3doy,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 360, numOutputBytes -> 7570)",null,Databricks-Runtime/17.3.x-scala2.13


Use this feature to safeguard data integrity and recover from unintended modifications.

## Conclusion
Delta Lake provides powerful features such as **Time Travel**, **Version History**, and **metadata management**, Delta Lake provides robust solutions for querying historical data, tracking changes, and recovering from unintended modifications. These capabilities not only ensure data reliability and consistency but also enhance scalability and performance.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>